# Laravel Core Deployment Package
Prepare upload.zip containing only the Laravel core application files so extraction places the contents directly into public_html/gp-admin/core/.

In [1]:
import os
import zipfile
from pathlib import Path
import fnmatch

In [2]:
root = Path(r'd:/Web App Projects/GP-Specialist/gp-admin').resolve()
output_zip = root / 'upload.zip'
core_items = [
    'app',
    'bootstrap',
    'config',
    'database',
    'resources',
    'routes',
    'storage',
    'vendor',
    'artisan',
    'composer.json',
    'composer.lock',
]

In [3]:
exclude_patterns = [
    '.git',
    '.gitignore',
    'node_modules',
    '*.log',
    '*.tmp',
    'storage/logs/*',
    'storage/framework/views/*',
    'storage/framework/sessions/*',
    'storage/framework/cache/*',
    'storage/framework/testing/*',
]

In [4]:
def should_exclude(rel_path: str) -> bool:
    rel_path = rel_path.replace('\\', '/').lstrip('./')
    for pattern in exclude_patterns:
        if fnmatch.fnmatch(rel_path, pattern):
            return True
    return False

if output_zip.exists():
    output_zip.unlink()

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item_name in core_items:
        item_path = root / item_name
        if item_path.is_file():
            zf.write(item_path, arcname=item_name)
        elif item_path.is_dir():
            for dirpath, dirnames, filenames in os.walk(item_path):
                rel_dir = Path(dirpath).relative_to(root).as_posix()
                for filename in filenames:
                    rel_file = f'{rel_dir}/{filename}' if rel_dir else filename
                    if should_exclude(rel_file):
                        continue
                    zf.write(Path(dirpath) / filename, arcname=rel_file)

print('Created upload.zip at', output_zip)

Created upload.zip at D:\Web App Projects\GP-Specialist\gp-admin\upload.zip


In [9]:
with zipfile.ZipFile(output_zip, 'r') as zf:
    entries = sorted(zf.namelist())
    print('Total entries:', len(entries))
    print('Sample contents:')
    for name in entries[:50]:
        print(name)

print('Vendor included:', any(name.startswith('vendor/') for name in entries))
print('Exact upload path on Hostinger/cPanel: public_html/gp-admin/core/')
print('Migration required after upload: likely no (unless database schema changed)')

Total entries: 18838
Sample contents:
app/Filament/Pages/Dashboard.php
app/Filament/Pages/Settings/GeneralSettingsPage.php
app/Filament/Pages/Settings/NotificationSettings.php
app/Filament/Pages/Settings/OtpSettingsPage.php
app/Filament/Pages/Settings/WorkflowSettingsPage.php
app/Filament/Resources/DiagnosticCenters/DiagnosticCenterResource.php
app/Filament/Resources/DiagnosticCenters/Pages/ManageDiagnosticCenterServices.php
app/Filament/Resources/DiagnosticCenters/Pages/ManageDiagnosticCenters.php
app/Filament/Resources/DiagnosticReferrals/DiagnosticReferralResource.php
app/Filament/Resources/DiagnosticReferrals/Pages/ManageDiagnosticReferrals.php
app/Filament/Resources/Gps/GpResource.php
app/Filament/Resources/Gps/Pages/ManageGps.php
app/Filament/Resources/Hospitals/HospitalResource.php
app/Filament/Resources/Hospitals/Pages/ManageHospitalSpecialists.php
app/Filament/Resources/Hospitals/Pages/ManageHospitals.php
app/Filament/Resources/Locations/LocationResource.php
app/Filament/Resou

In [6]:
# Rebuild upload.zip with refined exclusions to preserve empty storage directories and keep only Laravel core files
exclude_patterns = [
    'storage/logs/*',
    'storage/framework/views/*',
    'storage/framework/sessions/*',
    'storage/framework/cache/*',
    'storage/framework/testing/*',
    'node_modules/*',
]

def should_exclude(rel_path: str) -> bool:
    rel_path = rel_path.replace('\\', '/').lstrip('./')
    if rel_path.startswith('.git/') or rel_path == '.git':
        return True
    for pattern in exclude_patterns:
        if fnmatch.fnmatch(rel_path, pattern):
            return True
    return False

if output_zip.exists():
    output_zip.unlink()

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item_name in core_items:
        item_path = root / item_name
        if item_path.is_file():
            zf.write(item_path, arcname=item_name)
        elif item_path.is_dir():
            for dirpath, dirnames, filenames in os.walk(item_path):
                rel_dir = Path(dirpath).relative_to(root).as_posix()
                for filename in filenames:
                    rel_file = f'{rel_dir}/{filename}' if rel_dir else filename
                    if should_exclude(rel_file):
                        continue
                    zf.write(Path(dirpath) / filename, arcname=rel_file)

print('Rebuilt upload.zip at', output_zip)
print('Size:', output_zip.stat().st_size)


Rebuilt upload.zip at D:\Web App Projects\GP-Specialist\gp-admin\upload.zip
Size: 42321443


In [8]:
# Final rebuild: exclude runtime-generated files but preserve placeholder .gitignore files for storage directories
runtime_exclude = [
    'storage/logs/*',
    'storage/framework/views/*',
    'storage/framework/sessions/*',
    'storage/framework/cache/*',
    'storage/framework/testing/*',
]

placeholder_allow = [
    'storage/logs/.gitignore',
    'storage/framework/cache/.gitignore',
    'storage/framework/cache/data/.gitignore',
    'storage/framework/sessions/.gitignore',
    'storage/framework/testing/.gitignore',
    'storage/framework/views/.gitignore',
]

if output_zip.exists():
    output_zip.unlink()

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item_name in core_items:
        item_path = root / item_name
        if item_path.is_file():
            zf.write(item_path, arcname=item_name)
        elif item_path.is_dir():
            for dirpath, dirnames, filenames in os.walk(item_path):
                rel_dir = Path(dirpath).relative_to(root).as_posix()
                for filename in filenames:
                    rel_file = f'{rel_dir}/{filename}' if rel_dir else filename
                    if rel_file in placeholder_allow:
                        zf.write(Path(dirpath) / filename, arcname=rel_file)
                        continue
                    if any(fnmatch.fnmatch(rel_file, pattern) for pattern in runtime_exclude):
                        continue
                    if rel_file.startswith('.git/') or rel_file == '.git':
                        continue
                    if rel_file.startswith('node_modules/'):
                        continue
                    zf.write(Path(dirpath) / filename, arcname=rel_file)

print('Rebuilt upload.zip with final exclude rules at', output_zip)
print('Size:', output_zip.stat().st_size)


Rebuilt upload.zip with final exclude rules at D:\Web App Projects\GP-Specialist\gp-admin\upload.zip
Size: 42322408
